In [1]:
%%file app.py
from flask import Flask

app = Flask(__name__)

@app.route("/")
def home():
    return "Witaj w systemie monitoringu transakcji!"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Writing app.py


In [2]:
%%file app.py
from flask import Flask

app = Flask(__name__)

@app.route("/")
def home():
    return "Witaj w systemie monitoringu transakcji!"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [4]:
def score_transaction(tx: dict) -> dict:
    score = 0
    rules = []

    if tx.get("amount", 0) > 3000:
        score += 3; rules.append("R1: kwota > 3000")

    if tx.get("category") == "elektronika" and tx.get("amount", 0) > 1500:
        score += 2; rules.append("R2: elektronika > 1500")

    if tx.get("hour", 12) < 6:
        score += 2; rules.append("R3: nocna godzina")

    if score >= 5:
        risk_level = "CRITICAL"
    elif score >= 3:
        risk_level = "HIGH"
    elif score >= 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return {"score": score, "risk_level": risk_level, "triggered_rules": rules}

test_tx = {"tx_id": "TX001", "amount": 4500.0, "category": "elektronika", "hour": 3}
print(score_transaction(test_tx))

{'score': 7, 'risk_level': 'CRITICAL', 'triggered_rules': ['R1: kwota > 3000', 'R2: elektronika > 1500', 'R3: nocna godzina']}


In [5]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

def score_transaction(tx):
    score = 0
    rules = []
    if tx.get("amount", 0) > 3000:
        score += 3; rules.append("R1: kwota > 3000")
    if tx.get("category") == "elektronika" and tx.get("amount", 0) > 1500:
        score += 2; rules.append("R2: elektronika > 1500")
    if tx.get("hour", 12) < 6:
        score += 2; rules.append("R3: nocna godzina")
    risk_level = "CRITICAL" if score >= 5 else "HIGH" if score >= 3 else "MEDIUM" if score >= 1 else "LOW"
    return {"score": score, "risk_level": risk_level, "triggered_rules": rules}

@app.route("/score", methods=["POST"])
def score():
    tx = request.get_json()
    if not tx or "amount" not in tx:
        return jsonify({"error": "Brak pola 'amount'"}), 400
    result = score_transaction(tx)
    result["tx_id"] = tx.get("tx_id", "unknown")
    return jsonify(result)

@app.route("/health")
def health():
    return jsonify({"status": "ok", "version": "1.0-rules"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [7]:
import subprocess
import time
import requests

In [8]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

cases = [
    {"tx_id": "TX001", "amount": 50.0,   "category": "żywność",    "hour": 14},
    {"tx_id": "TX002", "amount": 1800.0,  "category": "elektronika", "hour": 10},
    {"tx_id": "TX003", "amount": 4500.0,  "category": "elektronika", "hour": 3},
]

for tx in cases:
    r = requests.post("http://localhost:5000/score", json=tx)
    res = r.json()
    print(f"{tx['tx_id']}  {tx['amount']:>7.0f} PLN  → {res['risk_level']:8s} (score={res['score']})  {res['triggered_rules']}")

TX001       50 PLN  → LOW      (score=0)  []
TX002     1800 PLN  → MEDIUM   (score=2)  ['R2: elektronika > 1500']
TX003     4500 PLN  → CRITICAL (score=7)  ['R1: kwota > 3000', 'R2: elektronika > 1500', 'R3: nocna godzina']


In [9]:
r = requests.post("http://localhost:5000/score", json={"tx_id": "TX000"})
print(r.status_code, r.json())

400 {'error': "Brak pola 'amount'"}


In [ ]:
1. GET pobiera dane, POST wysyła dane do przetwarzania

2. jsonify ustawia poprawny format JSON i nagłówki HTTP

3. Serwer obsłuży oba zapytania równolegle (kolejkowane / równoległe requesty)

In [10]:
server.kill()
print("Serwer zatrzymany.")

Serwer zatrzymany.
